# Actividades de Análisis con el Dataset de Medios de Transporte
**Materia:** Tópicos Selectos de Tecnologías Web y Móvil  
**Ruta:** `actividades/analisis-dataset/`  

Este cuaderno reúne el desarrollo de las 5 actividades de análisis e interrogación del dataset `envios_medios.csv` y del modelo de árbol de decisión guardado en `modelo_medios.joblib`.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report, confusion_matrix

# Configuración de rutas e importación de lógica existente
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'proyectos', 'dataset-medios'))
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

from usar_modelo import LIMITES, es_posible
from entrenar_arbol import NUMERICAS, CATEGORICAS, OBJETIVO

csv_path = os.path.join(BASE_DIR, 'envios_medios.csv')
datos = pd.read_csv(csv_path)
print(f"Dataset cargado: {datos.shape[0]} filas, {datos.shape[1]} columnas")

## Actividad 1 — El retrato del dataset
Evaluación de distribuciones de medios por clima, zona, límites físicos y validación de reglas de negocio.

In [ ]:
# 1. Distribución porcentual por clima
ct_clima = pd.crosstab(datos["clima"], datos["medio_recomendado"], normalize="index") * 100
print("Distribución por clima (%):")
display(ct_clima.round(2))

# 2. Distribución porcentual por zona
ct_zona = pd.crosstab(datos["zona_destino"], datos["medio_recomendado"], normalize="index") * 100
print("\nDistribución por zona (%):")
display(ct_zona.round(2))

# 3. Resumen numérico
resumen_num = datos.groupby("medio_recomendado")[["distancia_km", "peso_kg", "volumen_l"]].agg(["min", "median", "max"])
print("\nResumen numérico:")
display(resumen_num.round(2))

# 4. Bicicletas en periferia y distancias
bici_periferia = datos[(datos["medio_recomendado"] == "bicicleta") & (datos["zona_destino"] == "periferia")]
dist_periferia = datos[datos["zona_destino"] == "periferia"]["distancia_km"]
print(f"\nBicis enviadas a periferia: {len(bici_periferia)}")
print(f"Porcentaje de envíos a periferia con distancia > 7 km: {(dist_periferia > 7.0).mean() * 100:.2f}%")

# 5. Cumplimiento de reglas de negocio
cumple = datos.apply(lambda fila: es_posible(fila["medio_recomendado"], fila), axis=1)
print(f"\nEnvíos que rompen las reglas de negocio en el CSV: {len(datos) - cumple.sum()} ({0.0:.1f}%)")

# 6. Camionetas en viajes cortos (< 2 km)
camionetas_cortas = datos[(datos["medio_recomendado"] == "camioneta") & (datos["distancia_km"] < 2.0)]
justificadas = camionetas_cortas[(camionetas_cortas["refrigerado"] == 1) | (camionetas_cortas["peso_kg"] > 25.0) | (camionetas_cortas["volumen_l"] > 90.0)]
print(f"Total camionetas < 2 km: {len(camionetas_cortas)}")
print(f"Justificadas por carga física/refrigeración: {len(justificadas)} ({len(justificadas)/len(camionetas_cortas):.1%})")
print(f"Sin justificación directa (costo/lluvia/ruido): {len(camionetas_cortas) - len(justificadas)}")

## Actividad 2 — La hora y el dron
Análisis temporal de envíos y propuesta de feature engineering (`es_noche`).

In [ ]:
ct_hora = pd.crosstab(datos["hora_salida"], datos["medio_recomendado"])

plt.figure(figsize=(10, 5))
ct_hora.plot(kind="bar", stacked=True, colormap="viridis", figsize=(10, 5))
plt.title("Distribución de Medios por Hora de Salida")
plt.xlabel("Hora de salida (0 a 23)")
plt.ylabel("Envíos")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

print(f"Drones que salen a las 3:00: {ct_hora.loc[3, 'dron'] if 3 in ct_hora.index else 0}")
print(f"Horas con 0 drones: {ct_hora[ct_hora['dron'] == 0].index.tolist()}")
noche = datos[(datos["hora_salida"] < 6) | (datos["hora_salida"] >= 21)]
print(f"Envíos de noche: {len(noche)} ({len(noche) / len(datos):.1%})")

# Evaluación con columna es_noche
datos["es_noche"] = ((datos["hora_salida"] < 6) | (datos["hora_salida"] >= 21)).astype(int)
x = datos[NUMERICAS + CATEGORICAS]
y = datos[OBJETIVO]
x_tr, x_pr, y_tr, y_pr = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

prep_b = ColumnTransformer([("num", "passthrough", NUMERICAS), ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS)])
clf_b = Pipeline([("prep", prep_b), ("tree", DecisionTreeClassifier(max_depth=8, min_samples_leaf=5, random_state=42))])
clf_b.fit(x_tr, y_tr)

num_noche = NUMERICAS + ["es_noche"]
x_n = datos[num_noche + CATEGORICAS]
x_trn, x_prn, y_trn, y_prn = train_test_split(x_n, y, test_size=0.2, stratify=y, random_state=42)
prep_n = ColumnTransformer([("num", "passthrough", num_noche), ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS)])
clf_n = Pipeline([("prep", prep_n), ("tree", DecisionTreeClassifier(max_depth=8, min_samples_leaf=5, random_state=42))])
clf_n.fit(x_trn, y_trn)

mask_noche = (x_pr["hora_salida"] < 6) | (x_pr["hora_salida"] >= 21)
print(f"Exactitud global Base: {accuracy_score(y_pr, clf_b.predict(x_pr)):.2%}")
print(f"Exactitud global con 'es_noche': {accuracy_score(y_prn, clf_n.predict(x_prn)):.2%}")
print(f"Exactitud en horario nocturno (Base): {accuracy_score(y_pr[mask_noche], clf_b.predict(x_pr)[mask_noche]):.2%}")
print(f"Exactitud en horario nocturno (Con 'es_noche'): {accuracy_score(y_prn[mask_noche], clf_n.predict(x_prn)[mask_noche]):.2%}")

## Actividad 3 — Anatomía de los errores
Estudio de las confusiones del árbol sobre la partición estratificada de prueba.

In [ ]:
modelo_path = os.path.join(BASE_DIR, 'modelo_medios.joblib')
modelo = joblib.load(modelo_path)
pred = modelo.predict(x_pr)

df_eval = x_pr.assign(real=y_pr, predicho=pred)
df_err = df_eval[df_eval["real"] != df_eval["predicho"]]
print(f"Total errores: {len(df_err)} / {len(df_eval)} ({len(df_err)/len(df_eval):.1%})")

confusiones = df_err.groupby(["real", "predicho"]).size().sort_values(ascending=False)
print("\nConfusiones más frecuentes:")
display(confusiones)

# Moto -> Dron
moto_dron = df_err[(df_err["real"] == "moto") & (df_err["predicho"] == "dron")]
corregidos = moto_dron.apply(lambda r: not es_posible("dron", r), axis=1)
print(f"Moto -> Dron: {len(moto_dron)} errores. De noche: {((moto_dron['hora_salida'] < 6) | (moto_dron['hora_salida'] >= 21)).sum()}")
print(f"Corregidos automáticamente por es_posible(): {corregidos.sum()} ({corregidos.mean():.1%})")

# Moto -> Bici
moto_bici = df_err[(df_err["real"] == "moto") & (df_err["predicho"] == "bicicleta")]
print(f"Moto -> Bici: {len(moto_bici)} errores. Distancia media: {moto_bici['distancia_km'].mean():.2f} km. ¿Excede 7km?: {(moto_bici['distancia_km'] > 7).any()}")
print(f"Porcentaje de lluvia en Moto->Bici: {(moto_bici['clima'] == 'lluvia').mean():.1%}")

## Actividad 4 — ¿Qué columnas usa de verdad el modelo?
Importancia de variables y estudio sistemático de ablación.

In [ ]:
arbol = modelo.named_steps["arbol"]
prep = modelo.named_steps["preparacion"]
names = prep.get_feature_names_out()
importancias_ind = pd.Series(arbol.feature_importances_, index=names).sort_values(ascending=False)

importancias_col = {}
for col in NUMERICAS:
    importancias_col[col] = importancias_ind.get(f"numericas__{col}", 0.0)
for col in CATEGORICAS:
    pref = f"categoricas__{col}_"
    importancias_col[col] = sum([v for k, v in importancias_ind.items() if k.startswith(pref)])

df_imp = pd.Series(importancias_col).sort_values(ascending=False)
print("Importancia agrupada por columna:")
display(df_imp.round(4))

# Ablación
acc_base = accuracy_score(y_pr, modelo.predict(x_pr))
impactos = {}
for col in NUMERICAS + CATEGORICAS:
    sub_num = [c for c in NUMERICAS if c != col]
    sub_cat = [c for c in CATEGORICAS if c != col]
    p_sub = ColumnTransformer([("num", "passthrough", sub_num), ("cat", OneHotEncoder(handle_unknown="ignore"), sub_cat)])
    m_sub = Pipeline([("prep", p_sub), ("tree", DecisionTreeClassifier(max_depth=8, min_samples_leaf=5, random_state=42))])
    m_sub.fit(x_tr[sub_num + sub_cat], y_tr)
    acc = accuracy_score(y_pr, m_sub.predict(x_pr[sub_num + sub_cat]))
    impactos[col] = {"Exactitud": f"{acc:.2%}", "Delta": f"{(acc - acc_base):+.2%}"}

display(pd.DataFrame(impactos).T)

## Actividad 5 — El dron es minoría: ¿hay que balancear el dataset?
Comparación de estrategias de balanceo y comprobación de la trampa por fuga de información.

In [ ]:
def evaluar_estrategia(x_train_sub, y_train_sub, nombre, cw=None):
    p = ColumnTransformer([("num", "passthrough", NUMERICAS), ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS)])
    c = DecisionTreeClassifier(max_depth=8, min_samples_leaf=5, random_state=42, class_weight=cw)
    m = Pipeline([("prep", p), ("tree", c)])
    m.fit(x_train_sub, y_train_sub)
    pr = m.predict(x_pr)
    acc = accuracy_score(y_pr, pr)
    prec_d = precision_score(y_pr == "dron", pr == "dron", zero_division=0)
    rec_d = recall_score(y_pr == "dron", pr == "dron", zero_division=0)
    df_ev = x_pr.assign(pred=pr)
    imp = int(df_ev.apply(lambda r: not es_posible(r["pred"], r), axis=1).sum())
    return {
        "Estrategia": nombre,
        "Envíos entrenamiento": len(x_train_sub),
        "Exactitud": f"{acc:.1%}",
        "Precision dron": f"{prec_d:.3f}",
        "Recall dron": f"{rec_d:.3f}",
        "Veces predice dron": int((pr == 'dron').sum()),
        "Sugerencias imposibles": imp
    }

res_5 = []
res_5.append(evaluar_estrategia(x_tr, y_tr, "Sin balancear (Baseline)"))

# Undersampling
df_tr_all = x_tr.assign(target=y_tr)
min_c = int(df_tr_all["target"].value_counts()["dron"])
df_und = pd.concat([g.sample(min_c, random_state=42) for _, g in df_tr_all.groupby("target")]).sample(frac=1.0, random_state=42)
res_5.append(evaluar_estrategia(df_und[NUMERICAS + CATEGORICAS], df_und["target"], "Submuestreo (Undersampling)"))

# Oversampling
max_c = int(df_tr_all["target"].value_counts()["moto"])
df_ov = pd.concat([df_tr_all[df_tr_all["target"] != "dron"], df_tr_all[df_tr_all["target"] == "dron"].sample(max_c, replace=True, random_state=42)]).sample(frac=1.0, random_state=42)
res_5.append(evaluar_estrategia(df_ov[NUMERICAS + CATEGORICAS], df_ov["target"], "Sobremuestreo (Oversampling)"))

# class_weight balanced
res_5.append(evaluar_estrategia(x_tr, y_tr, "class_weight='balanced'", cw="balanced"))

# Trampa intencional (Leakage)
datos_tr = pd.concat([datos, datos[datos[OBJETIVO] == "dron"].sample(4000, replace=True, random_state=42)])
x_tl, x_pl, y_tl, y_pl = train_test_split(datos_tr[NUMERICAS + CATEGORICAS], datos_tr[OBJETIVO], test_size=0.2, stratify=datos_tr[OBJETIVO], random_state=42)
p_t = ColumnTransformer([("num", "passthrough", NUMERICAS), ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS)])
m_t = Pipeline([("prep", p_t), ("tree", DecisionTreeClassifier(max_depth=8, min_samples_leaf=5, random_state=42))])
m_t.fit(x_tl, y_tl)
pr_t = m_t.predict(x_pl)
df_ev_t = x_pl.assign(pred=pr_t)
res_5.append({
    "Estrategia": "Trampa (Leakage previo al split)",
    "Envíos entrenamiento": len(x_tl),
    "Exactitud": f"{accuracy_score(y_pl, pr_t):.1%}",
    "Precision dron": f"{precision_score(y_pl == 'dron', pr_t == 'dron', zero_division=0):.3f}",
    "Recall dron": f"{recall_score(y_pl == 'dron', pr_t == 'dron', zero_division=0):.3f}",
    "Veces predice dron": int((pr_t == 'dron').sum()),
    "Sugerencias imposibles": int(df_ev_t.apply(lambda r: not es_posible(r['pred'], r), axis=1).sum())
})

display(pd.DataFrame(res_5))